# Modulos

In [ ]:
# Librerías estándar de Python
import datetime  # Manejo de fechas y horas
import pandas as pd  # Manipulación y análisis de datos en DataFrames
import numpy as np  # Operaciones numéricas y manejo eficiente de arrays
import sys  # Acceso a variables y funciones del sistema
import re  # Expresiones regulares para procesamiento de texto
import os  # Operaciones del sistema de archivos

# Rutas

In [ ]:
# ============================================================================
# ⚙️ PARÁMETROS DE CONFIGURACIÓN MANUAL
# ============================================================================
# 🔧 ACTUALIZA ESTOS VALORES CADA EJECUCIÓN:

# 1. FECHA Y CARPETA
ANO = "2026"
MES_NUMERO = "03"  # Dos dígitos: "01", "02", etc.
MES_NOMBRE = "Marzo"  # Nombre del mes
DIA = "20"


# 2. ARCHIVOS DINÁMICOS (cambian cada ejecución)
ARCHIVO_MS_ADRES_EPSC25 = f"EPSC25MC00{DIA}{MES_NUMERO}{ANO}.TXT"  # Maestro Contributivo
ARCHIVO_MS_ADRES_EPS025 = f"EPS025MS00{DIA}{MES_NUMERO}{ANO}.TXT"  # Maestro Subsidiado


# ============================================================================
# DETECCIÓN AUTOMÁTICA DE RUTAS (UNIVERSAL)
# ============================================================================

# Obtener usuario del sistema automáticamente
usuario = os.environ.get('USERNAME') or os.environ.get('USER')
user_home = os.path.expanduser("~")

# Rutas base
ONEDRIVE_BASE = os.path.join(
    user_home, 
    "OneDrive - 891856000_CAPRESOCA E P S", 
    "Capresoca", 
    "AlmostClear"
)

ESCRITORIO = os.path.join(
    user_home,
    "OneDrive - 891856000_CAPRESOCA E P S",
    "Escritorio"
)

# Ruta del proyecto (universal)
PROYECTO_RAIZ = os.path.join(user_home, "Documents", "Proyectos Python", "capresoca-data-automation")
sys.path.append(os.path.abspath(PROYECTO_RAIZ))

# ============================================================================
# IMPORTAR MÓDULOS PERSONALIZADOS
# ============================================================================

from src.file_loader import cargar_maestros_ADRES  # Función para cargar archivos maestros ADRES
from src.data_cleaning import BduaReportProcessor  # Clase para limpiar y normalizar población Maestro ADRES
from src.data_cleaning import DataCleaner          # Clase para limpiar y normalizar DataFrames de Pandas

# ============================================================================
# CONSTRUCCIÓN DE RUTAS AUTOMÁTICAS
# ============================================================================

# --- RUTAS DE ENTRADA ---
R_Maestro_EPSC25 = os.path.join(ONEDRIVE_BASE, "Procesos BDUA", "Contributivo", "Maestro", ANO, ARCHIVO_MS_ADRES_EPSC25)
R_Maestro_EPS025 = os.path.join(ONEDRIVE_BASE, "Procesos BDUA", "Subsidiados", "Maestro", "MS", ANO, ARCHIVO_MS_ADRES_EPS025)


# ============================================================================
# VALIDACIÓN Y RESUMEN
# ============================================================================

print("=" * 80)
print(f"👤 USUARIO: {usuario}")
print(f"📁 OneDrive Base: {ONEDRIVE_BASE}")
print(f"📁 Proyecto: {PROYECTO_RAIZ}")
print("=" * 80)

print(f"\n📋 PARÁMETROS CONFIGURADOS:")
print(f"   📄 Maestro EPSC25: {ARCHIVO_MS_ADRES_EPSC25}")
print(f"   📄 Maestro EPS025: {ARCHIVO_MS_ADRES_EPS025}")


print("\n" + "-" * 80)
print("🔍 VALIDACIÓN DE RUTAS:")
print("-" * 80)

rutas_validar = {
    "📄 Maestro EPSC25": R_Maestro_EPSC25,
    "📄 Maestro EPS025": R_Maestro_EPS025,
}

errores = []
for nombre, ruta in rutas_validar.items():
    existe = os.path.exists(ruta)
    icono = "✅" if existe else "❌"
    print(f"{icono} {nombre}")
    print(f"   {ruta}")
    if not existe and "Carpeta de trabajo" not in nombre:
        errores.append(nombre)

if errores:
    print(f"\n⚠️ {len(errores)} ruta(s) no encontrada(s)")
    print("\n💡 Verifica:")
    print("   1. Que OneDrive esté sincronizado")
    print("   2. Que los parámetros al inicio estén actualizados")
    print("   3. Que los archivos existan en las carpetas")
    print("   4. Que el archivo 'Dataframe Pila' se haya generado previamente")
else:
    print(f"\n✅ Todas las rutas validadas correctamente")

print("=" * 80 + "\n")

# Dataframe

In [ ]:
maestro_ADRES = cargar_maestros_ADRES(R_Maestro_EPS025, R_Maestro_EPSC25)

# Listado censal o Sisben ADRES

In [ ]:
# Duplicar la columna "MARCASISBENIV+MARCASISBENIII_2" y nombrarla "MARCASISBENIV+MARCASISBENIII"
maestro_ADRES["MARCASISBENIV+MARCASISBENIII_2"] = \
    maestro_ADRES["MARCASISBENIV+MARCASISBENIII"]

# 1. Instanciar el procesador: Se crea un objeto pasando el DataFrame.
#    La jerarquía de población ya está definida por defecto dentro de la clase.
processor = BduaReportProcessor(df=maestro_ADRES)

# 2. Ejecutar la limpieza y asignarla de vuelta.
#    El método retorna un DataFrame completamente nuevo con la columna actualizada.
maestro_ADRES = processor.prioritize_population_markers(
    col_name="MARCASISBENIV+MARCASISBENIII"
)

# ¡Listo! 'maestro_ADRES' ahora contiene los datos limpios.